# Atlas — 70% split, seed consistency and held-out evaluation

Kept separate from the simulation notebook. Simulation has mispredicted this twice, so nothing here
should be inferred from there.

**Two traps, both of which caught us:**

1. **Matched-Ψ correlation is inflated.** 88% of Ψ's variance is the per-junction baseline every
   program shares, and matching is near-arbitrary — at K=30, 62% of programs match a second partner
   within 0.02 of the one chosen. A matched *r* of ~0.96 is nearly guaranteed. Prefer §4 (matching-free)
   and §5 (the biological output).
2. **A corrupt fit can look fine.** On 2026-07-29 four fits hit a non-finite loss, recovered ψ/φ/
   `psis_loc` via the checkpoint restore, exited 0, and left `psis_scale` 100% NaN. §2 checks first.

In [ ]:
import os, sys, json, glob, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt

FIGDIR = "/gpfs/commons/home/kisaev/Leaflet-analysis/GenomeBiology_Submission2026/figures"
sys.path.insert(0, FIGDIR)
import figure_common as fc
try:
    import leaflet_theme as th; th.apply_theme()
except Exception as e:
    print("theme unavailable:", e)

MODEL_DIR = f"{fc.ATLAS_WD}/LEAFLETFA_MODELS/2026-07-30_atlas9_default"
print("model dir:", MODEL_DIR)
print("fits present:", sum(os.path.exists(f"{MODEL_DIR}/mb_run_{t}/fit.npz") for t in range(9)), "/ 9")

## 1–2. Load, then the health gate

Do not skip the gate.

In [ ]:
FITS = fc.load_fits(MODEL_DIR)
H = fc.health(FITS)
print("!! DO NOT TRUST ANYTHING BELOW" if not H.attrs["all_ok"] else "all fits clean and stopped early")
H

In [ ]:
LABELS = fc.celltypes()
KEEP = [t for t, n in pd.Series(LABELS).value_counts().items() if n >= 50]
K = 30                                  # <-- change me
TASKS = [t for t in fc.TASKS_BY_K[K] if t in FITS]
print(f"K={K}, tasks {TASKS}, {len(KEEP)} cell types >=50 cells")

## 3. Is the matching decisive?

**Run before §4a.** If the margin is small, matched statistics are a coin-flip.

In [ ]:
rows = []
for a, b in fc.pairs(TASKS):
    _, matched, margin = fc.perm_from_psi(np.asarray(FITS[a]["psi"], float),
                                          np.asarray(FITS[b]["psi"], float))
    rows.append(dict(pair=f"{FITS[a]['seed']}v{FITS[b]['seed']}", matched_r=np.median(matched),
                     margin=np.median(margin), frac_ambiguous=np.mean(margin < 0.02)))
M = pd.DataFrame(rows); display(M.round(4))
print("matching NOT decisive -- prefer §4b and §5" if M.frac_ambiguous.mean() > 0.3 else "matching usable")

## 4a. Matched latents — interpret with §3 in hand

In [ ]:
rows = []
for a, b in fc.pairs(TASKS):
    A, B = FITS[a], FITS[b]
    col, matched, _ = fc.perm_from_psi(np.asarray(A["psi"], float), np.asarray(B["psi"], float))
    phiA = np.asarray(A["phi"], np.float32); phiB = np.asarray(B["phi"], np.float32)[:, col]
    st = max(1, phiA.shape[0] // 20000)
    pc = np.array([np.corrcoef(phiA[i], phiB[i])[0, 1] for i in range(0, phiA.shape[0], st)])
    pc = pc[np.isfinite(pc)]
    pa = np.asarray(A["psi"], np.float64); pb = np.asarray(B["psi"], np.float64)[col]
    ba = pa - (pa.sum(0, keepdims=True) - pa) / (K - 1)
    bb = pb - (pb.sum(0, keepdims=True) - pb) / (K - 1)
    rows.append(dict(pair=f"{A['seed']}v{B['seed']}", psi_matched=np.median(matched),
                     phi_percell=np.median(pc), phi_frac_gt_0p9=np.mean(pc > 0.9),
                     pi=np.corrcoef(np.asarray(A["pi"], float).ravel(),
                                    np.asarray(B["pi"], float).ravel()[col])[0, 1],
                     beta_slope=fc.effect_slope(ba, bb)))
pd.DataFrame(rows).round(4)

## 4b. Matching-free — quote these

In [ ]:
rows = []
for a, b in fc.pairs(TASKS):
    sp = fc.psi_subspace_agreement(FITS[a]["psi"], FITS[b]["psi"])
    sf = fc.cca(FITS[a]["phi"], FITS[b]["phi"])
    rows.append(dict(pair=f"{FITS[a]['seed']}v{FITS[b]['seed']}",
                     psi_canon=sp.mean(), psi_dirs=f"{(sp>0.9).sum()}/{K-1}",
                     phi_canon=sf.mean(), phi_dirs=f"{(sf>0.9).sum()}/{K-1}"))
pd.DataFrame(rows).round(4)

## 5. The biological output — which junctions are up in which cell types

**The claim to make.** No matching anywhere: cell-type labels are fixed.

In [ ]:
DELTA = 0.1                              # <-- try 0.05, 0.2
R = {t: fc.group_rate(FITS[t]["phi"], FITS[t]["psi"], LABELS, KEEP) for t in TASKS}
rows = []
for a, b in fc.pairs(TASKS):
    A, B = R[a], R[b]
    dA, dB = A - A.mean(0, keepdims=True), B - B.mean(0, keepdims=True)
    SA, SB = dA > DELTA, dB > DELTA
    rows.append(dict(pair=f"{FITS[a]['seed']}v{FITS[b]['seed']}",
                     rate_r=np.corrcoef(A.ravel(), B.ravel())[0, 1],
                     effect_slope=fc.effect_slope(dA, dB),
                     top1k=fc.topn_overlap(dA, dB, 1000),
                     top10k=fc.topn_overlap(dA, dB, 10000),
                     jaccard=(SA & SB).sum() / max((SA | SB).sum(), 1),
                     n_calls=int(SA.sum())))
pd.DataFrame(rows).round(4)

### What each number means

All five are computed from `rate[c, j] = (mean φ over cells of type c) @ Ψ[:, j]` — the model's
predicted splicing rate for cell type *c* at junction *j*. The **deviation** `dev[c,j] = rate[c,j] −
mean over cell types` is "junction *j* is elevated in cell type *c*", which is the biological
statement. Two fits are compared directly: cell-type labels are fixed, so **no program matching is
involved anywhere.**

| metric | definition | what it tells you | what would be bad |
|---|---|---|---|
| **predicted rate `r`** | Pearson correlation of the two fits' full `rate` matrices (n_celltypes × J, flattened) | do the fits predict the same splicing everywhere? | anything below ~0.95 — this is the most forgiving measure, since most junctions sit near the same baseline in both |
| **effect-size slope** | regression slope of fit B's `dev` on fit A's. **Not** the correlation | are the effect *magnitudes* the same? 1.0 = no attenuation, 0.9 = B's effects are 10% smaller | ≪1 means one fit systematically shrinks effects. **A correlation cannot see this** — it is scale-invariant and would read ~1.0 under any uniform shrinkage |
| **top-1,000 agree** | overlap of the 1,000 most-elevated (cell type, junction) pairs | do the fits pick the same *strongest* signals — the ones you would follow up? | low here is the most damaging failure: it means the headline hits differ between runs |
| **top-10,000 agree** | same, deeper into the list | does agreement hold past the very top? | a large drop from top-1,000 means only a handful of signals are stable |
| **Jaccard (effect > 0.1)** | intersection / union of the sets called at a fixed threshold | do the fits call the same *set* of differential junctions? | **expected to be the lowest number.** Thresholding a continuous quantity flips anything near the boundary, so this understates agreement — read it with top-N, not instead of it |

**Why Jaccard is lower than the rate correlation.** With ~118,000 calls per fit at δ = 0.1, a junction
whose deviation is 0.099 in one fit and 0.101 in the other counts as a complete disagreement. That is a
property of thresholding, not of the model. The top-N overlaps avoid it by ranking instead of cutting.

In [ ]:
# ── Recreate the K-comparison table exactly ────────────────────────────────────────
# Same computation as pipelines/05_leafletfa/A25_celltype_junction_reproducibility.py
DELTA = 0.1
rows = []
for KK, ts in fc.TASKS_BY_K.items():
    ts = [t for t in ts if t in FITS]
    if len(ts) < 2:
        continue
    Rk = {t: fc.group_rate(FITS[t]["phi"], FITS[t]["psi"], LABELS, KEEP) for t in ts}
    rate_r, slope, t1k, t10k, jac, ncalls = [], [], [], [], [], []
    for a, b in fc.pairs(ts):
        A, B = Rk[a], Rk[b]
        dA, dB = A - A.mean(0, keepdims=True), B - B.mean(0, keepdims=True)
        SA, SB = dA > DELTA, dB > DELTA
        rate_r.append(np.corrcoef(A.ravel(), B.ravel())[0, 1])
        slope.append(fc.effect_slope(dA, dB))
        t1k.append(fc.topn_overlap(dA, dB, 1000))
        t10k.append(fc.topn_overlap(dA, dB, 10000))
        jac.append((SA & SB).sum() / max((SA | SB).sum(), 1))
        ncalls.append(int(SA.sum()))
    rows.append({"K": KK, "predicted rate r": np.mean(rate_r), "effect-size slope": np.mean(slope),
                 "top-1,000 agree": np.mean(t1k), "top-10,000 agree": np.mean(t10k),
                 f"Jaccard (effect > {DELTA})": np.mean(jac), "calls/fit": int(np.mean(ncalls))})

TAB = pd.DataFrame(rows).set_index("K").T
display(TAB.style.format({c: "{:.4f}" for c in TAB.columns}))
best = TAB.loc[[i for i in TAB.index if i != "calls/fit"]].idxmax(axis=1)
print("best K per metric:", dict(best))
print(f"\nK={best.mode().iloc[0]} wins {int((best == best.mode().iloc[0]).sum())} of {len(best)} metrics")

## Co-clustering consistency (consensus clustering)

Cluster φ separately under each seed, then ask **how often a given pair of cells lands in the same
cluster**. This is the standard consensus-clustering construction (Monti et al., *Machine Learning*
2003): the **co-association matrix** `C[i,j] = fraction of runs placing i and j together`.

It is the right tool here for two reasons. It is **permutation-invariant** — cluster labels are
arbitrary in exactly the way program labels are, and asking about *pairs* removes the need to match
anything. And it degrades gracefully: a partition can shift its boundaries a little without the pair
structure collapsing, which a raw ARI would punish.

Three readouts:

| | meaning |
|---|---|
| **co-association distribution** | should be **bimodal** — pairs are either reliably together (≈1) or reliably apart (≈0). Mass in the middle is instability |
| **PAC** (Proportion of Ambiguous Clustering) | fraction of pairs with co-association strictly between 0.1 and 0.9 (Șenbabaoğlu et al., *Sci Rep* 2014). **Lower is better**; it is the standard scalar summary of that bimodality |
| **pairwise ARI** | agreement between the raw partitions. Reported alongside because it is what most papers quote, and because it is *harsher* than the pair view |

With only 3 seeds the co-association values can only be 0, 1/3, 2/3 or 1, so the histogram is coarse —
PAC counts the 1/3 and 2/3 pairs. More seeds would give a finer curve.

In [ ]:
_TASKS = TASKS
_PHI   = {t: np.asarray(FITS[t]["phi"], np.float32) for t in _TASKS}
_PHI_N = _PHI[_TASKS[0]].shape[0]
print(f"atlas: K={K}, {len(_TASKS)} seeds, {_PHI_N:,} cells")

In [ ]:
# ── consensus / co-clustering consistency ─────────────────────────────────────────
# N^2 pairs, so subsample. The E-step is per-cell independent, so this does not bias anything.
N_SUB   = 3000
METHOD  = "kmeans"        # "kmeans" (fast) or "leiden" (needs scanpy)
N_CLUST = 15              # kmeans only; leiden uses RESOLUTION
RESOLUTION = 1.0

rng = np.random.default_rng(0)
sub = np.sort(rng.choice(_PHI_N, size=min(N_SUB, _PHI_N), replace=False))

def cluster_phi(phi, method=METHOD, seed=0):
    X = np.nan_to_num(np.asarray(phi, np.float32))
    if method == "kmeans":
        from sklearn.cluster import KMeans
        return KMeans(n_clusters=N_CLUST, n_init=10, random_state=seed).fit_predict(X)
    import scanpy as sc, anndata as ad
    a = ad.AnnData(X)
    sc.pp.neighbors(a, n_neighbors=15, use_rep="X", random_state=seed)
    sc.tl.leiden(a, resolution=RESOLUTION, random_state=seed, key_added="l",
                 flavor="igraph", n_iterations=2, directed=False)
    return a.obs["l"].astype(str).values

parts = {t: cluster_phi(_PHI[t][sub], seed=0) for t in _TASKS}   # SAME clustering seed:
# only the model seed varies, so any disagreement comes from phi, not from the clusterer.
print({t: len(set(p)) for t, p in parts.items()}, "clusters per seed")

# co-association: fraction of runs putting each pair together
n = len(sub)
C = np.zeros((n, n), dtype=np.float32)
for p in parts.values():
    lab = pd.factorize(p)[0]
    C += (lab[:, None] == lab[None, :]).astype(np.float32)
C /= len(parts)
iu = np.triu_indices(n, 1)
co = C[iu]

PAC = float(np.mean((co > 0.1) & (co < 0.9)))
from sklearn.metrics import adjusted_rand_score as ari
aris = [ari(parts[a], parts[b]) for a, b in fc.pairs(_TASKS)]

print(f"\nPAC (ambiguous pairs, lower is better) : {PAC:.4f}")
print(f"pairs always together                  : {np.mean(co == 1):.4f}")
print(f"pairs never together                   : {np.mean(co == 0):.4f}")
print(f"pairwise ARI between partitions        : {np.mean(aris):.4f}  {np.round(aris,3).tolist()}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(co, bins=np.linspace(0, 1, len(parts) * 2 + 3), color="#006865")
ax[0].set_xlabel("co-association (fraction of seeds together)")
ax[0].set_ylabel("cell pairs"); ax[0].set_yscale("log")
ax[0].set_title(f"bimodal = stable   |   PAC = {PAC:.3f}")
o = np.argsort(pd.factorize(parts[_TASKS[0]])[0])
im = ax[1].imshow(C[np.ix_(o, o)], cmap="BrBG", vmin=0, vmax=1, interpolation="nearest")
ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].set_title("co-association matrix")
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

## 6. Cell-cell similarity across seeds

In [ ]:
N_SHOW = 600
rng = np.random.default_rng(0)
idx = np.sort(rng.choice(len(LABELS), size=N_SHOW, replace=False))
order = idx[np.argsort(pd.Categorical(LABELS[idx], categories=KEEP, ordered=True).codes)]
S = {t: fc.cellcell(np.asarray(FITS[t]["phi"], np.float32)[order]) for t in TASKS}
iu = np.triu_indices(len(order), 1)
agree = [np.corrcoef(S[a][iu], S[b][iu])[0, 1] for a, b in fc.pairs(TASKS)]
fig, ax = plt.subplots(1, len(TASKS), figsize=(4 * len(TASKS), 4.2))
for axi, t in zip(np.atleast_1d(ax), TASKS):
    im = axi.imshow(S[t], cmap="BrBG", vmin=-1, vmax=1)
    axi.set_title(f"seed {FITS[t]['seed']}"); axi.set_xticks([]); axi.set_yticks([])
fig.colorbar(im, ax=ax, fraction=0.02, label="cell-cell similarity")
fig.suptitle(f"K={K} — between-seed agreement r = {np.mean(agree):.3f}")
plt.show()

## 7. Held-out evaluation (imputation, structure)

Populated once the eval jobs finish.

In [ ]:
import glob
imp = sorted(glob.glob(f"{MODEL_DIR}/eval_grouped/task_*/rs_all/04e_allcov.csv"))
st  = sorted(glob.glob(f"{MODEL_DIR}/eval_grouped/task_*/07c_out.csv"))
print(f"imputation shards: {len(imp)}/9    structure shards: {len(st)}/9")
if imp:
    IMP = pd.concat([pd.read_csv(x) for x in imp])
    display(IMP.head())
if st:
    STR = pd.concat([pd.read_csv(x) for x in st])
    display(STR.head())

## 8. Sweep K

In [ ]:
out = []
for KK, ts in fc.TASKS_BY_K.items():
    ts = [t for t in ts if t in FITS]
    if len(ts) < 2: continue
    Rk = {t: fc.group_rate(FITS[t]["phi"], FITS[t]["psi"], LABELS, KEEP) for t in ts}
    rr, sl, tk, ph = [], [], [], []
    for a, b in fc.pairs(ts):
        A, B = Rk[a], Rk[b]
        dA, dB = A - A.mean(0, keepdims=True), B - B.mean(0, keepdims=True)
        rr.append(np.corrcoef(A.ravel(), B.ravel())[0, 1]); sl.append(fc.effect_slope(dA, dB))
        tk.append(fc.topn_overlap(dA, dB, 10000))
        s = fc.cca(FITS[a]["phi"], FITS[b]["phi"]); ph.append((s > 0.9).sum() / (KK - 1))
    out.append(dict(K=KK, rate_r=np.mean(rr), effect_slope=np.mean(sl),
                    top10k=np.mean(tk), phi_dirs=np.mean(ph)))
pd.DataFrame(out).round(4)

## Scratch

In [ ]:
#